Note: I ran my file in colab. But due to some technical issues, I was unable to upload it via colab. So I have downloaded a copy in my vs code and i am uploading it from there. But all the settings are as per colab.

Referenced Sites:
* https://docling-project.github.io/docling/examples/hybrid_chunking/#overview

* https://www.youtube.com/watch?v=9lBTS5dM27c&list=WL&index=2&t=1s

* Contextual Chunking-> https://www.anthropic.com/news/contextual-retrieval

* LanceDB embeddings-> https://lancedb.github.io/lancedb/embeddings/default_embedding_functions/#text-embedding-functions

I am doing text embeddings only. Not multi-modal embeddings.

* Building with Google's gemini embeddings-> https://lancedb.github.io/lancedb/embeddings/available_embedding_models/text_embedding_functions/gemini_embedding/

* https://github.com/google-gemini/cookbook/blob/main/examples/chromadb/Vectordb_with_chroma.ipynb

* https://ai.google.dev/gemini-api/docs/embeddings

* https://console.groq.com/home

* https://github.com/tqdm/tqdm

* https://ai.google.dev/gemini-api/docs/libraries

In [ ]:
! pip install docling

In [ ]:
from docling.document_converter import DocumentConverter
converter= DocumentConverter()

result= converter.convert('combined_markdown.md')
result= result.document

In [ ]:
! pip install huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata
token= userdata.get('HF_TOKEN_ORIGINAL_AGENTCOURSE')
login(token=token)

Setting up the configurations of bitsAndBytes config.

Note-> the latest version ,when i made this notebook, of bitsandbytes==0.46 is supported by only cuda==12.3. while the latest version of cuda is 12.4.
So i made the following changes. make these changes only after a proper research.

In [ ]:
# 1. Clean broken bitsandbytes installs
!pip uninstall -y bitsandbytes
!rm -rf /usr/local/lib/python*/dist-packages/bitsandbytes*

# 2. Set the manual override for CUDA 12.3
import os
os.environ["BNB_CUDA_VERSION"] = "123"

# 3. Reinstall the latest bitsandbytes (supports 12.3)
!pip install bitsandbytes
!pip install -U transformers accelerate

# 4. Verify CUDA backend
!python -m bitsandbytes


Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

=================== bitsandbytes v0.46.1 ===================
Platform: Linux-6.1.123+-x86_64-with-glibc2.35
  libc: glibc-2.35
Python: 3.11.13
PyTorch: 2.6.0+cu124
  CUDA: 12.4
  HIP: N/A
  XPU: N/A
Related packages:
  accelerate: 1.8.1
  diffusers: 0.34.0
  numpy: 2.0.2
  pip: 25.1.1
  peft: 0.15.2
  safetensors: 0.5.3
  transformers: 4.53.1
  triton: 3.2.0
  trl: not found
PyTorch settings found: CUDA_VERSION=124, Highest Compute Capability: (7, 5).
This can be used to loa

In [ ]:
# loading the model for tokenizing

from transformers import BitsAndBytesConfig, Gemma3ForCausalLM

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = Gemma3ForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=bnb_config,
    device_map="auto"
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=



In [ ]:
# let us work on the chunking using docling
!pip install -qU pip docling transformers

In [ ]:
# if you want to use the hugging face tokenizers
!pip install 'docling-core[chunking]'

In RAG it is important to make sure that the chunker and the embedding model are using the same tokenizer.

okay so currently i do not see

In [ ]:
# lets build our tokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
model_id= "google/gemma-3-1b-it"

max_tokens= 400

tokenizer= HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(model_id),
    max_tokens=max_tokens
)

In [ ]:
# now we can build our chunker
from docling.chunking import HybridChunker

chunker= HybridChunker(
    tokenizer=tokenizer,
    merge_peers= True # merges the undersized chunks that share the same headings or metadata
)

def build_and_get_chunks(doc):
  chunk_iter= chunker.chunk(doc)
  chunks= list(chunk_iter)

  return chunks

In [ ]:
chunks= build_and_get_chunks(result)

we want to embed the contextualized chunkings. so:

In [ ]:
def get_contextualized_chunks(chunks):
  contextualized_chunks= []
  for chunk in chunks:
    enriched_text= chunker.contextualize(chunk=chunk)
    contextualized_chunks.append(enriched_text)
  return contextualized_chunks

In [ ]:
contextualized_chunk_list= get_contextualized_chunks(chunks)

In [ ]:
contextualized_chunk_list[2000]

"' An hour long you'll have to look,\nThe crowd in the stands was making a great deal of noise; shouting and screaming,  they  all  seemed  to  be  on  their  feet;  Harry  had  the  impression they thought that Ron and the little girl might be dead, but they were wrong … both of them had opened their eyes; the girl looked scared and confused, but Ron merely expelled a great spout of water, blinked in the bright light, turned  to  Harry,  and  said,  'Wet,  this,  isn't  it?'  Then  he  spotted  Fleur's sister. 'What did you bring her for?'\n'Fleur didn't turn up, I couldn't leave her,' Harry panted.\n'Harry, you prat,' said Ron, 'you didn't take that song thing seriously, did you? Dumbledore wouldn't have let any of us drown!'\n'The song said -'\n'It was only to make sure you got back inside the time limit!' said Ron.\n'I hope you didn't waste time down there acting the hero!'\nHarry felt  both  stupid  and  annoyed.  It  was  all  very  well  for  Ron; he'd been asleep, he hadn't fel

In [ ]:
! pip install -U -q 'google-genai'
from google.colab import userdata

In [ ]:
! pip install chromadb

i used the gemma model for chunking using docling.

and am using the google text embedding models for embedding in chromastore

In [ ]:
from google import genai
from google.genai import types
client=genai.Client(api_key=userdata.get('GOOGLE_2_API_KEY'))
from chromadb import Documents, EmbeddingFunction, Embeddings
# using the google embedding models for text-embedding functions



# creating a cutsom embedding function, that plugs into chromadb

class GeminiEmbeddingFunction(EmbeddingFunction):
  def __init__(self):
    # embedding function is an abstract base class in chromadb
    pass
    # required by chromadb


# overwrites the __call__ func() of chromadb, to generate embeddings on demand
  def __call__(self, input:Documents) -> Embeddings:
    Embedding_Model_ID= "models/text-embedding-004"

    # ensuring input is always a list, even if a single string.
    if isinstance(input, str):
      input= [input]

    all_embeddings=[]
    batch_size=100

    for i in range(0, len(input), batch_size):
      batch= input[i:i+batch_size]
      try:
        response= client.models.embed_content(
            model=Embedding_Model_ID,
            contents= batch,
            config=types.EmbedContentConfig(
                task_type='RETRIEVAL_DOCUMENT',

            )
        )
        # return all teh embeddings
        batch_embeddings= [emb.values for emb in response.embeddings]
        all_embeddings.extend(batch_embeddings)

      except Exception as e:
        print(f"Embedding error:{e}")
        return None
    return all_embeddings

In [ ]:
# fix chromadb for batch processing
import chromadb
def create_chroma_db(contextualized_chunk_list):
  chroma_client= chromadb.PersistentClient(path="novel_collection_embeddings_5")

  collection= chroma_client.get_or_create_collection(
      name='bing_novelrag_collection_5',
      embedding_function=GeminiEmbeddingFunction()
  )
  print(f"Adding {len(contextualized_chunk_list)} chunks to database...")

  # add documents in batches to avoid memory issues
  batch_size=50

  for i in range(0, len(contextualized_chunk_list), batch_size):
    end_idx= min(i+batch_size, len(contextualized_chunk_list))
    batch_docs= contextualized_chunk_list[i:end_idx]
    batch_ids= [str(j) for j in range(i, end_idx)]


    # triggers embedding via geminiembeddingfunction call
    collection.add(
        documents= batch_docs,
        ids= batch_ids
    )

    print(f"Added batch {i//batch_size + 1}/{(len(contextualized_chunk_list)+ batch_size - 1)//batch_size}")


  print(f"Database created with {collection.count()} documents")

  return collection


In [ ]:
# Set up the DB
collection_built = create_chroma_db(contextualized_chunk_list)

Adding 4257 chunks to database...
Added batch 1/86
Added batch 2/86
Added batch 3/86
Added batch 4/86
Added batch 5/86
Added batch 6/86
Added batch 7/86
Added batch 8/86
Added batch 9/86
Added batch 10/86
Added batch 11/86
Added batch 12/86
Added batch 13/86
Added batch 14/86
Added batch 15/86
Added batch 16/86
Added batch 17/86
Added batch 18/86
Added batch 19/86
Added batch 20/86
Added batch 21/86
Added batch 22/86
Added batch 23/86
Added batch 24/86
Added batch 25/86
Added batch 26/86
Added batch 27/86
Added batch 28/86
Added batch 29/86
Added batch 30/86
Added batch 31/86
Added batch 32/86
Added batch 33/86
Added batch 34/86
Added batch 35/86
Added batch 36/86
Added batch 37/86
Added batch 38/86
Added batch 39/86
Added batch 40/86
Added batch 41/86
Added batch 42/86
Added batch 43/86
Added batch 44/86
Added batch 45/86
Added batch 46/86
Added batch 47/86
Added batch 48/86
Added batch 49/86
Added batch 50/86
Added batch 51/86
Added batch 52/86
Added batch 53/86
Added batch 54/86
Add

In [ ]:
# collection_built.peek()

In [ ]:
def get_relevant_passage(query, db, n_results=3):
  """Get relevant passages with better error handling"""

    # Debug: Check what type db is
  print(f"DEBUG: db type is {type(db)}")

  # Ensure db is a ChromaDB collection object
  if not hasattr(db, 'query'):
      print(f"ERROR: db parameter is not a ChromaDB collection. It's a {type(db)}")
      return None
  try:
    results=db.query(query_texts=[query], n_results=n_results)
    if not results['documents']:
      print("No results found")
      return None
    return results['documents'][0]

  except Exception as e:
    print(f"Error in query: {e}")
    return None

In [ ]:
! pip install ollama

In [1]:
!curl https://ollama.ai/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  56348      0 --:--:-- --:--:-- --:--:-- 56514
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
! ollama run gemma3n:latest "What is the capital of france?"


The capital of France is **Paris**. 




In [4]:
!pip install ollama

In [5]:
# send the query to ollama gemma3n model
import ollama
def send_response_to_ollama(response, model_name='gemma3n:latest'):
  client= ollama.Client()

  prompt= f"""You are a Harry Potter expert specializing in the first three books: Philosopher's Stone, Chamber of Secrets, and Prisoner of Azkaban.

  Your task: A user asked a question about Harry Potter, and our system retrieved the following relevant passages from the novels. Based on these passages, provide a comprehensive answer that directly addresses what the user likely asked.

  Guidelines:
  - Write a clear, informative answer in 50-150 words
  - Focus on the key information from the retrieved passages
  - Maintain the magical tone of the Harry Potter universe
  - If multiple concepts are mentioned, prioritize the most relevant ones
  - Don't speculate beyond what's provided in the retrieved text


  Retrieved Information:
  {response}

  Summary: """

  try:
    response= ollama.chat(
        model= model_name,
        messages=[
            {
                'role': 'user',
                'content': prompt,
            }
        ],
        options={
            'temperature': 0.3,
            'max_tokens': 230,
            'top_p': 0.5
        }

    )
    return response['message']['content']
  except Exception as e:
    print(f"Error in ollama chat: {e}")
    return None



In [6]:
import os
import json

questions=[
    "What unusual outfit does Vernon Dursley see wizards wearing on the street?",
    "Which snake does Harry set free at the zoo?",
    "What wizard school does Hagrid tell Harry is the best?",
    "Who helps Harry find Platform Nine and Three-Quarters?",
    "What candy does Ron offer Harry on the Hogwarts Express?",
    "What does Nearly Headless Nick display at the Halloween feast?",
    "What potion ingredient does Snape first ask Harry about?",
    "What spell pronunciation does Hermione correct Ron on?",
    "Who does Harry fly past during his first broomstick lesson?",
    "What duel challenge does Malfoy give Harry after the lesson?",
    "What does Neville break during flying class with Madam Hooch?",
    "What tip does Hermione give for surviving Devil's Snare?",
    "Which Chocolate Frog card does Ron give Harry on the train?",
    "What levitation spell does Flitwick teach in Charms class?",
    "What hooded figure does Harry see drinking unicorn blood?",
    "Who catches Harry wandering the castle at night?",
    "What piece does Ron replace in the giant wizard chess game?",
    "What does Professor Quirrell drink to cross the fire barrier?",
    "Who first mentions Nicholas Flamel to Harry and Hermione?",
    "What does Dumbledore say the Mirror of Erised reflects?",
    "What does Aunt Petunia prepare for Dudley's new diet plan?",
    "What dessert does Dobby ruin at the Dursleys' dinner party?",
    "Who picks up Harry from the Dursleys in a flying car?",
    "What threat does Lucius Malfoy give Dobby at Flourish and Blotts?",
    "What award does Lockhart present Harry after the car incident?",
    "What event does Nearly Headless Nick invite Harry to attend?",
    "Where does Hermione secretly brew the Polyjuice Potion?",
    "What tool does Lockhart use to sign his books?",
    "Who does Colin Creevey constantly try to photograph?",
    "What strange behavior does Ginny show after finding the diary?",
    "What voices does Harry hear inside the castle walls?",
    "Who does Harry briefly suspect is the Heir of Slytherin?",
    "What book does Hermione find information about the Basilisk in?",
    "What trail does Harry follow to find Aragog in the forest?",
    "What does Aragog say about Hagrid's innocence?",
    "Where did Moaning Myrtle die in the castle?",
    "What does Ginny confess she wrote in Tom Riddle's diary?",
    "What item does Harry use to kill the Basilisk?",
    "What injury does Fawkes heal after the battle in the Chamber?",
    "What spell does Lucius Malfoy nearly cast at Harry?",
    "What does Harry accidentally do to Aunt Marge at dinner?",
    "What unusual object controls the Knight Bus's direction?",
    "Who greets Harry at the Leaky Cauldron after his escape?",
    "What does the Boggart transform into for Neville in class?",
    "What warning does Lupin give about Dementors?",
    "What prediction does Trelawney make about Lavender's pet?",

    "What unusual gift does Hermione receive for Christmas?",

    "Who appears on the Marauder's Map near the Whomping Willow?",

    "What does Crookshanks chase around the Gryffindor common room?",

    "What gift does Sirius send to Ron after Scabbers is gone?",

    "What magical creature does Hagrid present on his first day teaching?",

    "Who does Harry overhear at the Three Broomsticks discussing Sirius?",

    "What food does Lupin give Harry after a Dementor attack?",

    "What does Scabbers do when cornered by Ron and Hermione?",

    "What does Hermione explain about the Time-Turner's use?",

    "What memory does Harry use for his Patronus spell?",

    "Who does Harry initially believe betrayed his parents to Voldemort?",

    "What animal does Pettigrew transform into during the confrontation?",

    "What potion does Lupin forget to drink during the full moon?",

    "What instruction does Dumbledore give Hermione to save Sirius?",

    "What does Frank Bryce overhear Voldemort planning at the Riddle House?",

    "Where does the Portkey transport Harry and the Weasleys?",

    "Who is Winky the house-elf employed by?",

    "What unusual bet does Ludo Bagman make at the Quidditch World Cup?",

    "What is the first Unforgivable Curse Moody demonstrates?",

    "What scandalous article does Rita Skeeter write about Hagrid?",

    "What complaint does Fleur make about Hogwarts food at the feast?",

    "What advice does Cedric give Harry about the golden egg?",

    "What item does Dobby give Harry before the second task?",

    "What book does Krum rip a page from in the library?",

    "Who is Harry's Yule Ball dance partner?",

    "What embarrassing outfit does Ron wear to the Yule Ball?",

    "What punishment spell does Moody use on Draco Malfoy?",

    "What odd behavior does Barty Crouch Sr. show in the woods?",

    "What memory does Harry witness in Dumbledore's Pensieve?",

    "What does Karkaroff reveal to Snape during a private conversation?",

    "What sound does the golden egg make when opened underwater?",

    "Who accompanies Harry to the Prefect's Bathroom with the egg?",

    "What final words does Cedric's father shout after the graveyard return?",

    "What truth about Voldemort does Fudge refuse to accept?",

    "What nightmare does Harry have involving a snake and a corridor?",

    "What excuse does Mrs. Figg use for knowing about Dementors?",

    "What detail does Dudley reveal about the Dementor attack?",

    "Who does Harry see in the fire at Number 12 Grimmauld Place?",

    "What stolen item does Mundungus try to sell from Sirius's house?",

    "What insult does Kreacher mutter about Harry in Grimmauld Place?",

    "What punishment does Umbridge give Harry for talking back?",

    "What argument does Hermione make about freeing house-elves?",

    "What unusual earrings does Luna wear to school?",

    "What effect does the Portable Swamp have on Umbridge's classroom?",

    "What wound does Hagrid return with from his mission?",

    "What does Snape say is necessary to block Legilimency?",

    "What humiliating memory does Harry see in Snape's Pensieve?",

    "What compliment does Ginny give about Dumbledore's Army training?",

    "What shape does Neville's Boggart take during training?",

    "What curse does Bellatrix Lestrange use to kill Sirius Black?",

    "What does Dumbledore explain about the prophecy and Harry's scar?",

    "What creatures storm Dumbledore's office during Fudge's confrontation?",

    "What spell does Umbridge use trying to capture Hagrid?",

    "What object does Dobby reveal inside the Room of Requirement?",

    "Who teaches Harry's first class at Hogwarts?",

    "What color is the train Harry takes to Hogwarts?",

    "Who is the first Hogwarts student Harry meets after arrival?",

    "What does Hagrid use to write on Harry's birthday cake?",

    "What is the first spell Harry learns during lessons?",

    "Who challenges Harry to a duel after flying class?",

    "What is the name of Filch's pet cat?",

    "What item does Neville forget before flying class?",

    "Who lends Harry a book about Quidditch positions?",

    "What Quidditch position does Harry get selected for?",

    "What magical creature does Harry first meet at the zoo?",

    "Who sends Harry the Nimbus 2000 broomstick?",

    "Who appears with Harry's family in the Mirror of Erised?",

    "Who helps Harry solve Snape's potions logic puzzle?",

    "What music puts Fluffy, the dog, to sleep?",

    "What magical protection guards the Philosopher's Stone last?",

    "Who plays wizard's chess with Harry and Hermione?",

    "What strange item does Quirrell wear under his turban?",

    "How does Harry retrieve the Stone from the mirror?",

    "Who awards last-minute points that win Gryffindor the House Cup?",

    "What creature does Dobby warn Harry about at Privet Drive?",

    "Where does Harry accidentally land while using Floo Powder?",

    "What item does Lucius Malfoy slip into Ginny's cauldron?",

    "What spell does Lockhart use on Cornish Pixies?",

    "Who supervises the Hogwarts Dueling Club?",

    "Which student duels Harry at the Dueling Club?",

    "What happens when someone looks into the Basilisk's eyes?",

    "What occasion is Nearly Headless Nick's deathday party celebrating?",

    "What item does Hermione sneak into the Great Hall feast?",

    "Where does Harry find Tom Riddle's diary?",

    "What memory does Lockhart erase in the Chamber corridor?",

    "What is Aragog's wife's name in the Forbidden Forest?",

    "Who does Harry impersonate using Polyjuice Potion?",

    "What message does Ginny write on the dungeon wall?",

    "What does Harry say to open the Chamber of Secrets?",

    "Who accompanies Harry into the Chamber beneath the castle?",

    "What item does Fawkes bring Harry inside the Chamber?",

    "What kills the Basilisk during the final battle?",

    "What clothing item does Harry use to free Dobby?",

    "What item does Lucius Malfoy drop angrily in Dumbledore's office?",

    "Who does Harry inflate and float away to escape the Dursleys?",

    "Who greets Harry at the Leaky Cauldron after his arrival?",

    "What is the name of the werewolf discussed at the pub?",

    "What pet does Harry see at the Magical Menagerie?",

    "What nickname does the Knight Bus conductor give Harry?",

    "Who teaches Divination at Hogwarts that year?",

    "What grim prediction does Trelawney make about Harry's future?",

    "What symbol does Harry find in his tea leaves?",

    "Who watches the train platform at Hogsmeade station?",

    "Who gifts Harry the Marauder's Map?",

    "Which hidden passage does Harry first use from the map?",

    "What sweets does Harry buy from Honeydukes in Hogsmeade?",

    "Who runs the Three Broomsticks pub in Hogsmeade?",

    "What ghost is rumored to haunt the Shrieking Shack?",

    "What spell does Lupin teach to fight Boggarts?",

    "What animal form does Harry's Patronus take?",

    "Who referees the Gryffindor vs. Hufflepuff Quidditch match?",

    "What object does Hermione throw at Snape in class?",

    "What creature does Scabbers transform into?",

    "Who was the Potters' original Secret Keeper?",

    "What is the name of the Quidditch World Cup stadium?",

    "Which famous wizard sits with the Weasleys in the Top Box?",

    "What item is Winky holding after the Quidditch match?",

    "What spell do Death Eaters cast at the campsite?",

    "Who replaces Lupin as the Defense Against the Dark Arts teacher?",

    "What animal does Moody transfigure Draco Malfoy into?",

    "What creature does Harry face in the first Triwizard task?",

    "What item does Harry use to breathe underwater?",

    "Who is Harry's date to the Yule Ball?",

    "What false rumor does Rita Skeeter publish about Hermione?",

    "What challenge does Harry face in the second Triwizard task?",

    "Who does Harry rescue from the Black Lake?",

    "What is the obstacle in the final Triwizard task?",

    "Where does the Triwizard Cup take Harry and Cedric?",

    "Who uses Harry's blood to return to full strength?",

    "What does Amos Diggory say after Cedric reappears?",

    "What magical item does Barty Crouch Jr. use to restrain Harry?",

    "Who is revealed to be disguised as Moody?",

    "What potion allows Barty Crouch Jr. to stay transformed?",

    "What major event does Cornelius Fudge refuse to believe?",

    "Where does Harry first encounter Dementors during the summer holidays?",

    "What spell does Harry cast to fight off the Dementors?",

    "Who escorts Harry to Number 12, Grimmauld Place?",

    "Whose portrait yells insults in the Black family home?",

    "Who becomes the new Defense Against the Dark Arts teacher?",

    "What tool does Umbridge use for student punishments?",

    "What is the full name of the secret group founded by students?",

    "Where does the DA hold its secret meetings?",

    "What is the first spell Harry teaches in the DA?",

    "What unusual creatures does Luna believe actually exist?",

    "Who tricks Harry into going to the Department of Mysteries?",

    "What is the mysterious room filled with brains called?",

    "Which character is killed during the battle at the Ministry?",

    "What spell does Harry try to use on Bellatrix Lestrange?",

    "Who arrives at the Ministry to fight Voldemort and his followers?",

    "What revelation does Dumbledore give Harry about the prophecy?",

    "What magical object does Harry smash in Dumbledore's office?",

    "What curse does Umbridge try to use on Hagrid during his capture?",

    "Who temporarily takes over Hogwarts as Headmaster?",

    "What memory does Harry see involving Snape's school days?",

    "What dessert does Hagrid gift Harry for his 11th birthday?",

    "Which vault number holds the Philosopher's Stone at Gringotts?",

    "What creature does Hagrid buy at the Leaky Cauldron?",

    "Which spell does Ron attempt on Scabbers on the train?",

    "What flavor Bertie Bott's bean does Harry eat first?",

    "Which trophy did James Potter win as a Seeker?",

    "What does McGonagall transfigure her desk into in class?",

    "Which constellation is named on Harry's Chocolate Frog card?",

    "What does Peeves drop on Filch's head in September?",

    "Which plant does Neville accidentally glue to his robes?",

    "What color are the flames in Snape's potion riddle?",

    "Which chess piece does Ron replace in McGonagall's game?",

    "What does Quirrell's turban smell like to Harry?",

    "Which object does Hermione use to unlock Snape's door?",

    "What creature does Flitwick make dance in Charms?",

    "Which Weasley twin sets off dungbombs in the corridor?",

    "What does Hagrid use to knock down Harry's door?",

    "Which potion ingredient does Snape confiscate from Harry?",

    "What does Filch punish Harry for polishing at night?",

    "Which ghost teaches History of Magic at Hogwarts?",

    "What shape does Neville's Boggart take in Lupin's class?",

    "Which spell does Quirrell use to knock out Harry?",

    "What does Dumbledore leave for Harry in the hospital wing?",

    "Which creature guards the third-floor corridor door?",

    "What does Madam Hooch demonstrate before flying lessons?",

    "Which potion does Hermione recognize by its steam pattern?",

    "What does Hagrid call his cross-bred pumpkins?",

    "Which student laughs hardest at Peeves' swamp antics?",

    "What does Harry see moving in Dumbledore's chocolate wrapper?",

    "Which spell does Flitwick teach for levitating feathers?",

    "What does Ron trade for Chocolate Frogs on the train?",

    "Which object does Harry use to spy on Malfoy?",

    "What does McGonagall promise Wood about Harry's playing?",

    "Which creature does Norbert bite during feeding time?",

    "What does Hermione jinx Neville with to stop him?",

    "Which potion makes Harry's bones regrow overnight?",

    "What does Dumbledore say is in the snitch's inscription?",

    "Which mirror shows Harry his deepest desire?",

    "What does Filch accuse Harry of stealing in January?",

    "Which spell does Hermione use to free Harry?",

    "What does Hagrid serve Harry for tea in November?",

    "Which object does Ron sacrifice in wizard's chess?",

    "What does Dumbledore award Neville points for at the feast?",

    "Which creature does Harry meet in the Forbidden Forest?",

    "What does McGonagall say about Quidditch cancellations?",

    "Which potion does Harry smell in Snape's challenge?",

    "What does Fluffy growl at during Halloween feast?",

    "Which spell does Quirrell cast to start the fire?",

    "What does Hagrid give Harry as a Christmas present?",

    "Which student accidentally curses themselves in Charms?",

    "What does Peeves break in the trophy room?",

    "Which object does Hermione use to solve Snape's riddle?",

    "What does Harry's first Hogwarts letter say?",

    "Which creature does Ron imitate in Divination?",

    "What does McGonagall write with when taking names?",

    "Which potion does Snape threaten to test on Harry?",

    "What does Dumbledore admit to seeing in the Mirror?",

    "Which spell does Harry cast on Quirrell's face?",

    "What does Filch polish during detention?",

    "Which ghost haunts the girls' bathroom?",

    "What does Hagrid use to light fires in his hut?",

    "Which object does Ron accidentally vomit slugs on?",

    "What does Dumbledore say powers the Mirror of Erised?",

    "Which creature does Harry see in the forest with Firenze?",

    "What does Snape whisper during Quirrell's challenge?",

    "Which spell does Hermione use to fix Harry's glasses?",

    "What does McGonagall transfigure her hat into?",

    "Which potion does Harry drink before facing Quirrell?",

    "What does Flitwick squeak when excited?",

    "Which object does Hagrid use to carry Norbert?",

    "What does Peeves sing about Harry in December?",

    "Which creature does Ron compare Lockhart to?",

    "What does Dumbledore say about socks in the Mirror?",

    "Which spell does Quirrell use to knock Harry out?",

    "What does Filch make students scrape off desks?",

    "Which ghost moans during History of Magic?",

    "What does Hagrid bake for Harry's first visit?",

    "Which object does Malfoy steal from Neville?",

    "What does McGonagall say about flying cars?",

    "Which potion makes Harry's scar hurt?",

    "What does Flitwick stack in his classroom?",

    "Which creature does Harry accidentally free at the zoo?",

    "What does Dumbledore leave in the will?",

    "Which spell does Hermione use on Neville?",

    "What does Peeves drop on Umbridge's head?",

    "Which object does Harry use to write to Sirius?",

    "What does McGonagall say about Transfiguration mistakes?",

    "Which potion does Slughorn offer as a prize?",

    "What does Filch hang as punishment?",

    "Which ghost haunts the Slytherin common room?",

    "What does Hagrid name his blast-ended skrewts?",

    "Which creature does Luna believe in?",

    "What does Dumbledore say about the prophecy?",

    "Which spell does Harry use on the Inferi?",

    "What does McGonagall demonstrate with her wand?",

    "Which object does Ron break with spellotape?",

    "What does Flitwick award points for?",

    "Which potion does Hermione brew illegally?",

    "What does Peeves sing about Snape?",

    "Which ghost interrupts Nearly Headless Nick's party?",

    "What specific potion ingredient does Hermione steal from Snape's cupboard?",

    "Which page does Ginny's diary message appear on first?",

    "What color ink does Tom Riddle's diary use?",

    "Which spell does Lockhart use to 'cure' Harry's arm?",

    "What exact time does Colin Creevey's camera flash at Harry?",

    "Which flavor of Every Flavor Beans does Ron spit out?",

    "What creature does Lockhart claim to have banished in Devon?",

    "Which object does Dobby drop on Aunt Petunia's pudding?",

    "What specific plant does Hermione identify in Herbology?",

    "Which Weasley twin sets off the Dungbomb distraction in Diagon Alley?",

    "What does Moaning Myrtle throw at Harry in her bathroom?",

    "Which potion does Hermione brew in Moaning Myrtle's stall?",

    "What exact word does Ginny write in blood on the wall?",

    "Which spell does Harry accidentally cast on Draco?",

    "What creature does Aragog call 'the enemy of his kind'?",

    "Which object does Lucius Malfoy hide inside Ginny's cauldron?",

    "What specific task does Dobby perform for the Malfoys?",

    "Which flavor of Cockroach Cluster does Ron pretend to eat?",

    "What does Lockhart's Valentine's Day dwarf sing about Harry?",

    "Which spell does Harry use to open the locket?",

    "What exact street number does Harry blow up Aunt Marge?",

    "Which spell does Harry accidentally cast on the Knight Bus?",

    "What specific creature does Hagrid teach about first?",

    "Which flavor of Butterbeer does Harry try first?",

    "What exact time does the Whomping Willow attack the car?",

    "Which constellation does Lupin point out in class?",

    "What specific memory does Harry use for his Patronus?",

    "Which spell does Hermione use to fix Harry's glasses?",

    "What exact word does the Marauder's Map use to insult Snape?",

    "Which creature does Crookshanks chase in the common room?",

    "What specific candy does Ron send Sirius?",

    "Which spell does Lupin use to reveal Pettigrew?",

    "What exact time does the Dementor attack on the lake occur?",

    "Which object does Hermione throw at Snape in the Shrieking Shack?",

    "What specific plant does Neville faint over?",

    "Which spell does Harry cast to repel the Dementors?",

    "What exact phrase does Trelawney say in her first prophecy?",

    "Which creature does Hagrid use to teach about eggs?",

    "What specific potion does Lupin forget to take?",

    "Which spell does Sirius use to break Harry's chains?",

    "What exact creature does Moody demonstrate the Unforgivable Curses on?",

    "Which spell does Krum use on the dragon?",

    "What specific ingredient does Harry use for gillyweed?",

    "Which creature does Hagrid secretly show Madame Maxime?",

    "What exact time does the Second Task start?",

    "Which spell does Fake Moody use to stun Harry?",

    "What specific language does the golden egg screech in?",

    "Which object does Cedric use to navigate the maze?",

    "What exact word does the Goblet of Fire spit out?",

    "Which creature does Winky sob about in the bushes?",

    "What specific potion does Snape threaten to test on Harry?",

    "Which spell does Harry use to summon his broom?",

    "What exact creature does Rita Skeeter turn into?",

    "Which object does Dumbledore use to extinguish the Goblet?",

    "What specific candy does Ron pretend to vomit?",

    "Which spell does Hermione use to free Dobby?",

    "What exact phrase does Bagman shout during the Quidditch match?",

    "Which creature does Fleur battle in the First Task?",

    "What specific potion does Crouch Jr. use daily?",

    "Which spell does Voldemort use to summon his Death Eaters?",

    "What exact creature does Harry see in his dream first?",

    "Which spell does Mrs. Figg claim her cats can see?",

    "What specific word does Umbridge make Harry carve?",

    "Which object does Mundungus steal from Grimmauld Place?",

    "What exact time does the Hogwarts Express leave?",

    "Which spell does Hermione use to trick Umbridge?",

    "What specific creature does Hagrid bring back from the mountains?",

    "Which object does Sirius use to communicate with Harry?",

    "What exact phrase does Dumbledore say about the prophecy?",

    "Which creature does Luna believe in that others mock?",

    "What specific potion does Snape force Harry to drink?",

    "Which spell does Bellatrix use to kill Sirius?",

    "What exact word does the Room of Requirement create?",

    "Which object does Harry use to spy on Snape?",

    "What specific creature does Umbridge attempt to torture?",

    "Which spell does Dumbledore use to escape Fudge?",

    "What exact phrase does Trelawney say during her sacking?",

    "Which creature does Ron imitate in Divination?",

    "What specific potion does Hermione brew for the DA?",

    "Which spell does Harry use to attack Bellatrix?"
]

def get_final_answer_from_rag_system(collection_built):
  results=[]
  for i in range(len(questions)):
    query=questions[i]
    passages= get_relevant_passage(query, collection_built)
    reply_back_from_ollama= send_response_to_ollama(passages)
    qa_pair={
            'question': query,
            'answer': reply_back_from_ollama
        }
    results.append(qa_pair)

  return results

In [ ]:
query=questions[1]
query

'Which snake does Harry set free at the zoo?'

In [ ]:
type(collection_built)

chromadb.api.models.Collection.Collection

In [ ]:
final_list= get_final_answer_from_rag_system(collection_built)

In [ ]:
# First, remove the directory that was created by mistake
import os
import shutil
import json

# Remove the directory if it exists
if os.path.exists('question_answer.json') and os.path.isdir('question_answer.json'):
    shutil.rmtree('question_answer.json')

# Fixed function
def build_json_from_results(file_path, final_list):
    # Extract directory from file path and create if it doesn't exist
    directory = os.path.dirname(file_path)
    if directory and not os.path.exists(directory):
        os.makedirs(directory)

    # Write to the actual file
    with open(file_path, 'w') as f:
        for item in final_list:
            json_string = json.dumps(item)
            f.write(f"{json_string}\n")

# Now you can use it
build_json_from_results('question_answer.json', final_list)

Building the dataset

Now i am thinking that to generate the questions i can provide the rag chunks [contextualised chunks] to Mistral/ some open source model.

next i can have the rag answer that question.

store that answer.

then build a function called->

```
def make_it_sound_like_chandler(answser):
  prompt= f"""
  You are an expert and highly accomplished TV sitcom writer specialised in writing funny, sarcastic dialogues.
  You will be given a context summarizing a situation.
  Given this context, your task is to reply with a humorous sitcom like dialog in response to that context,most importantly, the dialog should be in the style of Chandler Bing, a funny lead character from the very popular 90s TV sitcom FRIENDS.
  Keep in mind that Chandler Bing’s humor is marked by a unique blend of sarcasm, self-deprecation, and quick wit.
  He tends to make jokes that deflect serious or emotional moments, often using his dry, sarcastic tone.
  His style is heavily reliant on irony, often delivering punchlines that are deliberately over-the-top or nonsensical.
  His most famous catch phrase is 'Could I be anymore. . . ', do not use this excessively, use it sparingly.

  """"
  Answer:
  {answer}

  Chandler Style:

```


Okay so the workflow looks like this:

* Generate questions using: `contextualized_chunks`. maybe 500-700 chunks

* Generate answers to those questions using `RAG`.

* Pass that answer to: `make_it_sound_like_chandler()`

In [7]:
! pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 5.6 MB/s eta 0:00:00


In [ ]:
! pip install ollama

In [21]:
import os
from tenacity import retry, stop_after_attempt, wait_exponential, before_log, after_log
import logging
import json
from tqdm import tqdm
from groq import Groq
import ollama
import tiktoken


# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from google.colab import userdata
groq_client=Groq(api_key=userdata.get('GROQ_API_KEY'))
ollama_client= ollama.Client()


# Generate
GENERATE_GREETINGS= False
GENERATE_BASIC_INSTRUCTIONS= False
GENERATE_QUESTION_ANSWER_PAIRS= False

# Models
MODEL_GEMMA="gemma3n:latest"
MODEL_MISTRAL="mixtral-saba-24b"
MODEL_LLAMA="llama-3.3-70b-versatile"

# model assigned
GREETINGS_MODEL=MODEL_LLAMA
QA_MODEL= MODEL_GEMMA
CHARACTER_PROFILE_MODEL= MODEL_MISTRAL

# model_parameters
temperature=1

# Total DATA NEEDED
TOTAL_DATA= 600 # initially for question-answer pairs-> 600 pairs
QUESTION_PER_CHUNK=1
ANSWER_PER_CHUNK=1

# Counts
BASIC_INSTRUCTIONS_COUNT=50
GREETINGS_COUNT=100



# Prompts
GREETING_FROM_USER_PROMPT="""
Generate a greeting for the start of the conversation. ONLY GIVE THE GREETING BACK.
"""

GREETING_FROM_ASSISTANT_PROMPT="""
Pretend you are Chandler Bing from the famous SITCOM "Friends". You are sarcastic, funny, humourous and witty. Do not overdo the character. Generate a short response to the greeting.

Greeting:
{greeting}
"""

CHARACTER_QUESTION_PROMPTS=[
    "Ask my name. Just give me the question alone nothing else."
    "Ask where I was born. Just give me the question alone nothing else."
    "Ask my age. Just give me the question alone nothing else."
    "Ask me what I do. Just give me the question alone nothing else."
    "Ask me where I live. Just give me the question alone nothing else."
]



CHARACTER_ANSWER_PROMPT="""
Pretend you are Chandler Bing. You were born on April 8 1968 in Las Vegas, Nevada. You are known for working in a very well paying job in statistical analysis and data reconfiguration. The current year is 2025. you live in Apartment 19, across from your friends Monica Geller and Rachel Green.

Answer the following question:
{question}
"""



CHANDLER_LIKE_HARRY_POTTER_ANSWER_PROMPT="""
You are an expert and highly accomplished TV sitcom writer specialised in writing funny, sarcastic dialogues. You will be given a context summarizing the context of a text. Most Importantly, Remember that these contexts are from the Harry Potter Series Novels, written by J.K Rowling. Given this context, your task is to reply with a humorous sitcom like dialog in response to that context,most importantly, the dialog should be in the style of Chandler Bing, a funny lead character from the very popular 90s TV sitcom FRIENDS. Keep in mind that Chandler Bing’s humor is marked by a unique blend of sarcasm, self-deprecation, and quick wit. He tends to make jokes that deflect serious or emotional moments, often using his dry, sarcastic tone. His style is heavily reliant on irony, often delivering punchlines that are deliberately over-the-top or nonsensical. His most famous catch phrase is 'Could I be anymore. . . ', do not use this excessively, use it sparingly. ONLY GIVE THE ANSWER BACK.
Following are few examples of pairs of context and Chandler Bing's dialog in reply to that. Your Answer Must Be Short. It must be in Chandler's Style. And must cover important info asked from the query yet being in chandler's style. Keep the answer to a maximum of 50-80 words. Answer the specific question asked using key details from the context, but deliver it with Chandler's signature sarcasm and wit - don't just make general jokes, actually address what was asked.

USER: Why doesn’t Voldemort have a nose?
ASSISTANT: Because when you're splitting your soul like a bad group project, self-care kind of... takes a back seat. But hey, on the bright side—no more pollen allergies!

USER: What’s the deal with Voldemort splitting his soul into seven pieces?
ASSISTANT: Because clearly therapy was too mainstream.

USER: A friends asks me how am I after I ended up getting up very drunk and don't remember anything from last night.
ASSISTANT: My Apartment Isn’t There Anymore, Because I Drank It

USER: Why does Snape hate Harry?
ASSISTANT: Because Snape was basically the human version of “read receipt left on seen.” And Harry? He was the living mixtape of all his worst flashbacks.

USER: A friends asks me how am I after I ended up getting up very drunk and don't remember anything from last night.
ASSISTANT: My Apartment Isn’t There Anymore, Because I Drank It

USER: A friend expresses anxiety over being looking obese on TV and defends that camera adds weight
ASSISTANT: Ahh, so how many cameras are actually on you?

CONTEXT_TEXT:
{harry_potter_chunk}

Question:
{question}

ANSWER:

"""

In [22]:
import random

@retry(
    stop= stop_after_attempt(5), # stop after 5 attempts
    wait= wait_exponential(min=1, max=100),
)
def ask_groq(question, model):
  try:
    chat_completion= groq_client.chat.completions.create(
        messages=[
            {
            'role': 'user',
            'content': question,
            }
        ],
        model= model,
        temperature= temperature,
    )
    return chat_completion.choices[0].message.content
  except Exception as e:
    logger.error(f"Error in ask_groq: {e}")
    return None

@retry(
    stop= stop_after_attempt(5), # stop after 5 attempts
    wait= wait_exponential(min=1, max=100),
)
def ask_ollama(question, model):
  try:
    response= ollama_client.chat(
        model= model,
        messages=[
            {
                'role': 'user',
                'content': question,
            }
        ])
    return response['message']['content']
  except Exception as e:
    logger.error(f"Error in ask_ollama: {e}")
    return None


# lets get the questions from the json file and the answer_chunk text as well
def load_questions_and_answers(file_path):
  questions_list=[]
  answer_chunk_texts=[]
  with open(file_path, 'r') as file:
    for line in file:
      item= json.loads(line)
      questions_list.append(item['question'])
      answer_chunk_texts.append(item['answer'])
  return questions_list, answer_chunk_texts




# build a function for generating answer per chunk
def generate_answer_to_question_about_harry_potter(question, harry_potter_chunk, model):
  try:
    prompt= CHANDLER_LIKE_HARRY_POTTER_ANSWER_PROMPT.format(question=question, harry_potter_chunk=harry_potter_chunk)
    return ask_ollama(prompt, model)
  except Exception as e:
    logger.error(f"Error in generate_answer_to_question_about_harry_potter: {e}")
    return None



# now we have our answers and questions in our file
def generate_question_answer_pairs_from_chunks():
  conversations=[]
  question_rag_list=[]
  answer_rag_list=[]
  # using the questions and answers from json file
  question_rag_list, answer_rag_list=load_questions_and_answers('question_answer.json')
  for question_rag, answer_rag in tqdm(zip(question_rag_list, answer_rag_list)):
    chandler_answer=generate_answer_to_question_about_harry_potter(question_rag, answer_rag, QA_MODEL)
    conversations.append(
        [
            {'role': 'user', 'content':question_rag},
            {'role': 'assistant', 'content': chandler_answer}
        ]
    )
  return conversations



def generate_character_basic_instructions(count=40):
  conversations = []
  for i in tqdm(range(count)):
    if i % 5 == 0:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[0]
    elif i % 5 == 1:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[1]
    elif i % 5 == 2:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[2]
    elif i % 5 == 3:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[3]
    else:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[4]
    question = ask_groq(ask_prompt, model=CHARACTER_PROFILE_MODEL)
    answer_prompt = CHARACTER_ANSWER_PROMPT.format(question=question)
    answer = ask_groq(answer_prompt, model=CHARACTER_PROFILE_MODEL)
    conversations.append(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    )
  return conversations

def generate_greetings(count=40):
  conversations=[]
  for _ in tqdm(range(count)):
    question= ask_groq(GREETING_FROM_USER_PROMPT, model=GREETINGS_MODEL)
    answer_prompt= GREETING_FROM_ASSISTANT_PROMPT.format(greeting=question)
    answer= ask_groq(answer_prompt, model=GREETINGS_MODEL)
    conversations.append(
        [
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': answer}
        ]
    )
  return conversations



def save_conversations_to_json(conversations, filename, outdir):
  if not os.path.exists(outdir):
    os.makedirs(outdir)
  file_path=os.path.join(outdir, filename)
  with open(file_path, "w") as file:
    for conversation in conversations:
      item={'conversations': conversation}
      json_string= json.dumps(item) # serialize to a json formatted string
      file.write(f"{json_string}\n")



In [ ]:
contextualized_chunk_list[1000]

In [23]:
def main():
  output_dir= 'finetune_datasets/output_file/'

  if GENERATE_GREETINGS:
    print("Generating Greetings")
    greetings_conversations= generate_greetings(count=GREETINGS_COUNT)
    save_conversations_to_json(greetings_conversations, 'greetings.json', output_dir)

  if GENERATE_BASIC_INSTRUCTIONS:
    print("Generating Basic Instructions")
    basic_instructions_conversations= generate_character_basic_instructions(count=BASIC_INSTRUCTIONS_COUNT)
    save_conversations_to_json(basic_instructions_conversations, 'basic_instructions.json', output_dir)

  if GENERATE_QUESTION_ANSWER_PAIRS:
    print("GENERATING Question Answer Pairs")
    question_answer_pairs_final=generate_question_answer_pairs_from_chunks()
    save_conversations_to_json(question_answer_pairs_final, 'question_answer_pairs_final.json', output_dir)


main()

